In [32]:
from src.data_loader import load_training_data
from src.preprocessing import notch_filter, bandpass_filter, highpass_filter
import config
import numpy as np
import scipy
from math import log, e


In [4]:
import pywt

Functions

In [33]:
def entropy2(labels, base=None):
  """ Computes entropy of label distribution. """

  n_labels = len(labels)

  if n_labels <= 1:
    return 0

  value,counts = np.unique(labels, return_counts=True)
  probs = counts / n_labels
  n_classes = np.count_nonzero(probs)

  if n_classes <= 1:
    return 0

  ent = 0.

  # Compute entropy
  base = e if base is None else base
  for i in probs:
    ent -= i * log(i, base)

  return ent

In [5]:
edf_file = "../data/training/R1.edf"
xml_file = "../data/training/R1.xml"
multi_channel_data, labels, info = load_training_data(
            edf_file, xml_file
        )

Loading training data from ../data/training/R1.edf and ../data/training/R1.xml...


c:\Users\Ines\Documents\2025\KTH\Studies\Signal Processing and Data Analysis\Project\CM2013\Python\src\data_loader.py:56: RuntimeWarning: Invalid measurement date encountered in the header.
  raw = mne.io.read_raw_edf(edf_file_path, preload=True, verbose=False)


Identified channels:
  EEG: ['EEG(sec)', 'EEG']
  EOG: ['EOG(L)', 'EOG(R)']
  EMG: ['EMG']
  EEG: 2 channels, 3750 samples/epoch, 125.0 Hz
  EOG: 2 channels, 3750 samples/epoch, 125.0 Hz
  EMG: 1 channels, 3750 samples/epoch, 125.0 Hz

Loaded 1083 epochs (9.03 hours)
Sleep stage distribution:
  Wake: 332 epochs (30.7%)
  N1: 47 epochs (4.3%)
  N2: 457 epochs (42.2%)
  N3: 145 epochs (13.4%)
  REM: 102 epochs (9.4%)


In [7]:
# Extracting single signal
signal = multi_channel_data["eeg"][0][1]

In [24]:
len(signal) # The signal is one epoch, which we will be extracting the spectral features from

3750

Check Wavelets

In [22]:
pywt.families()
pywt.wavelist('coif')



['coif1',
 'coif2',
 'coif3',
 'coif4',
 'coif5',
 'coif6',
 'coif7',
 'coif8',
 'coif9',
 'coif10',
 'coif11',
 'coif12',
 'coif13',
 'coif14',
 'coif15',
 'coif16',
 'coif17']

Testing PyWavelet module

In [8]:
# Daubechies coefficient
db1 = pywt.Wavelet('db1')

In [10]:
c = pywt.wavedec(signal, db1)

In [16]:
num_of_coeffs = len(c)
num_of_coeffs

12

In [19]:
# Number of levels in the decomposition
pywt.dwt_max_level(len(signal), db1)

11

In [34]:
# Try extracting the entropy, and other statistics from every coefficient if possible
for coeff in c:
    # Do the feature extraction here for every coeff
    energy = np.sum(coeff**2)
    
    print(f"Energy: {energy}")

    # Get statistical moments
    # Mean
    mean = np.mean(coeff)
    print(f"Mean: {mean}")

    # Standard Deviation
    stdev = np.std(coeff)
    print(f"Standard deviation: {stdev}")

    # Skewness
    skewness = scipy.stats.skew(coeff) 
    print(f"Skewness: {skewness}")

    # Kurtosis
    kurt = scipy.stats.kurtosis(coeff)
    print(f"Kurtosis: {kurt}")

    # Entropy
    entropy = entropy2(coeff)
    print(f"Entropy: {entropy}")







Energy: 2.6847299754000974e-08
Mean: -8.110930725375133e-05
Standard deviation: 8.273409305610988e-05
Skewness: 0.0
Kurtosis: -2.0
Entropy: 0.6931471805599453
Energy: 2.2655072005058196e-08
Mean: 0.00010270812651977406
Standard deviation: 2.7902988179175055e-05
Skewness: -7.988605391462577e-16
Kurtosis: -2.0
Entropy: 0.6931471805599453
Energy: 5.671575573397258e-08
Mean: 6.208639705882355e-05
Standard deviation: 0.000101608160271443
Skewness: 0.8285044360233955
Kurtosis: -0.9131490986153108
Entropy: 1.3862943611198906
Energy: 3.7909258908833174e-08
Mean: -4.5320691980461335e-05
Standard deviation: 5.1814015883892795e-05
Skewness: -0.702045226640301
Kurtosis: -1.0378766518022122
Entropy: 2.0794415416798357
Energy: 1.379149883758652e-07
Mean: 2.585784313725488e-06
Standard deviation: 9.585221060504502e-05
Skewness: 0.6048584819978272
Kurtosis: 0.05237886854066387
Entropy: 2.70805020110221
Energy: 1.3353107128267983e-07
Mean: 1.3908166468191294e-05
Standard deviation: 6.525027674206248e-0

In [ ]:
# Measure the time it takes to preprocess the whole edf file